# 🔁 Day1_5 혼자 다시 정리 — Gradio 빈칸 채우기

`[Day1_5]실습.ipynb`를 복습하는 노트북입니다.

**사용법**
1. 코드 셀의 `___1___`, `___2___` … 같은 **빈칸**을 직접 채워서 실행합니다.
2. 막히면 바로 아래 **💡 힌트**를 보고, 그래도 안 되면 맨 아래 **✅ 정답** 섹션을 확인하세요.
3. 각 Step 끝의 **✍️ 개념 퀴즈**는 셀에 직접 답을 적어보세요.

| Step | 주제 |
|---|---|
| 1 | `gr.Interface` |
| 2 | `gr.ChatInterface` + 제곱 봇 |
| 3 | `gr.Blocks` 3요소 |
| 4 | 컴포넌트 6가지 + 조용한 버그 |
| 5 | `Row` / `Column` / `Tab` |
| 6 | `.click` + `.submit` |
| 7 | LangChain 결합 패턴 1·2·3 |
| 8 | 도전: 페르소나 + 메모리 + 초기화 |

# 🛠 Step 0. 환경 준비 (이 셀은 빈칸 없음)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
import gradio as gr

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
print("✅ 키 로드:", bool(os.environ.get("OPENAI_API_KEY")), "| Gradio", gr.__version__)

c:\Users\user\AppData\Local\pypoetry\Cache\virtualenvs\langchain-web-chatbot-N-6PPVO5-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 키 로드: True | Gradio 6.29.1


---
# 1️⃣ Step 1. `gr.Interface` — 함수 하나를 UI 하나로

```
[입력 컴포넌트] → [내 Python 함수] → [출력 컴포넌트]
```

**빈칸**
1. 이전에 띄운 Gradio 서버를 모두 정리하는 함수 (포트 충돌 방지)
2. 호출할 함수를 넘기는 인자 이름
3. 입력 컴포넌트 (텍스트박스)
4. UI를 실제로 띄우는 메서드

In [2]:
def hello(name):
    return f"안녕하세요, {name}님! 반가워요 👋"

gr.close_all()          # 빈칸 1
# 현재 열려있는 모든 포트 닫기. 이를 통해 포트 충돌 방지
demo = gr.Interface(
    fn = hello,                                          # 빈칸 2
    inputs=gr.Textbox(label="이름", placeholder="홍길동"),   # 빈칸 3
    outputs="text",
    title="인사 봇",
    description="이름을 입력하면 인사해줍니다.",
)

demo.launch()        # 빈칸 4

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


> 💡 **힌트**: 1 `close_...` / 2 "function"의 줄임말 / 3 텍스트 입력 컴포넌트 / 4 "실행·출시"

**✍️ 개념 퀴즈** — `inputs="text"` 와 `inputs=gr.Textbox(label=...)` 의 차이는?

답: 첫번째는 CLI 환경에서 값을 Text로 넣는 것이고 두번째는 그라디오에서 제공하는 textbox 컴포넌트 활용

---
# 2️⃣ Step 2. `gr.ChatInterface` — 챗봇 UI 한 줄

`chat_fn`의 약속: **인자가 (___1___, ___2___)** 두 개여야 한다. (history를 안 써도 반드시 있어야 함)

### 2-1. 메아리 봇

In [ ]:
def echo(message, history):         # 빈칸 1, 2 : 새 메시지 / 대화 내역
    return f"방금 '{message}'라고 하셨네요!"

gr.close_all()

demo = gr.Chatbot(                   # 빈칸 3 : 챗봇 UI 클래스
    fn=echo,
    title="🦜 메아리 봇",
)
demo.launch()

> 💡 **힌트**: 1 사용자가 방금 보낸 것 / 2 Gradio가 화면용으로 관리하는 기록 / 3 `Chat...`

### 2-2. 제곱 봇 (실습 1)
숫자를 입력하면 제곱을 돌려주고, 숫자가 아니면 안내 문구를 반환합니다.

In [ ]:
def sque(message, history):
    try:                                   # 빈칸 1 : 예외 처리 시작 키워드
        num = int(message)                 # 빈칸 2 : 문자열 → 정수 변환
        return f"입력하신 {num}의 제곱은 {num**2}이네요."   # 빈칸 3 : 거듭제곱 연산자
    except ValueError:                        # 빈칸 4 : 예외 잡기 키워드
        return "정수를 입력해주세요."

gr.close_all()
demo = gr.ChatInterface(fn=sque, title="🤞 제곱 봇")
demo.launch()

**✍️ 개념 퀴즈** — `"abc"` 를 입력했을 때 `int("abc")`가 던지는 에러 이름은? 그리고 왜 `try/except`가 필요한가?

답: int()는 정수를 만드는 함수이기 때문에 문자열 str을 입력하면 valueError 발생. 근데 소수의 경우는? 똑같이 valueError 나옴?

---
# 3️⃣ Step 3. `gr.Blocks` — 자유 레이아웃

### 🔑 Blocks의 핵심 3가지
1. `with gr.___1___() as demo:` — 이 안에 컴포넌트 정의
2. 컴포넌트는 **___2___에 저장** — `input_box = gr.Textbox(...)`
3. **이벤트 연결** — `btn.___3___(fn=..., inputs=..., outputs=...)`

(위 문장 빈칸 1~3을 먼저 아래에 적어보세요)

답: 1) Blocks 2) ______ 3) click
2) 번은 모르곘다!!

In [ ]:
gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 🎯 첫 Blocks 앱")

    input_box  = gr.Textbox(label="입력")
    output_box = gr.Textbox(label="출력", interactive=False)   # 빈칸 2 : 사용자가 직접 수정 못 하게
    btn        = gr.Button("복사하기", variant="primary")   # 빈칸 3 : 버튼 색상 옵션 이름

    # 버튼 클릭 → input 값을 output에 그대로 전달
    btn.click(fn=lambda x: x, inputs=input_box, outputs=output_box)   # 빈칸 4,5,6

demo.launch()

> 💡 **힌트**: 2 "interactive" / 3 "variant" / 5·6 위에서 만든 **변수 이름**

**✍️ 개념 퀴즈** — `inputs`/`outputs`에는 컴포넌트의 *값*이 아니라 무엇을 넘기나요?

답: 매개변수를 넘겨 gr.textbox에 입력된 값을 input으로 넘김.

---
# 4️⃣ Step 4. 컴포넌트 6가지 + 🐛 조용한 버그

| 컴포넌트 | 용도 |
|---|---|
| `Textbox` | 텍스트 입력/출력 |
| `Dropdown` | 페르소나·모델 선택 |
| `Slider` | temperature, max_tokens |
| `Radio` | 3~4개 중 하나 |
| `Checkbox` | True/False |
| `Button` | 클릭 트리거 |

In [ ]:
gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 🧱 컴포넌트 갤러리")

    txt = gr.Textbox(label="질문 입력", placeholder="여기에 입력하세요...", ___1___=2)    # 빈칸 1 : 줄 수

    persona = gr.___2___(                                   # 빈칸 2 : 드롭다운
        ___3___=["분석가", "마케터", "친구"],                 # 빈칸 3 : 선택지 목록 인자
        value="친구",
        label="페르소나 선택",
    )

    temperature = gr.___4___(minimum=0, maximum=2, step=0.1, value=0.7, label="Temperature")  # 빈칸 4

    style = gr.___5___(choices=["짧게", "보통", "자세히"], value="보통", label="답변 스타일")   # 빈칸 5

    use_stream = gr.___6___(value=True, label="스트리밍 사용")  # 빈칸 6

    btn = gr.Button("전송", variant="primary")   # primary=파랑, secondary=회색, ___7___=빨강  (빈칸 7)

    out = gr.Textbox(label="현재 설정", interactive=False, lines=5)

    def show_settings(t, p, temp, s, u):
        return f"질문: {t}\n페르소나: {p}\nTemp: {temp}\n스타일: {s}\n스트림: {u}"

    btn.click(
        fn=show_settings,
        inputs=[txt, persona, temperature, style, use_stream],   # 입력 순서 = 함수 ___8___ 순서 (빈칸 8)
        outputs=out,
    )

demo.launch()

> 💡 **힌트**: 4 숫자 조절 막대 / 5 둥근 선택 버튼 / 6 체크 표시 / 7 "멈춤" 의미 / 8 매개변수

### 🐛 조용한 버그 — 에러는 없는데 결과가 이상하다
아래 코드는 이름·나이를 입력하면 **"30님은 진환살입니다"** 처럼 나옵니다. `inputs=` 의 빈칸을 채워 고치세요.

In [ ]:
def show_user(name, age):
    return f"{name}님은 {age}살입니다."

gr.close_all()

with gr.Blocks() as demo:
    name_box = gr.Textbox(label="이름")
    age_box  = gr.Number(label="나이", value=30)
    out      = gr.Textbox(label="결과", interactive=False)
    btn      = gr.Button("확인", variant="primary")

    btn.click(show_user, inputs=[___1___, ___2___], outputs=out)   # 빈칸 1, 2 : 순서 주의!

demo.launch()

**✍️ 개념 퀴즈** — 이 버그가 "조용한" 이유는? 이런 종류의 버그를 잡으려면 어떤 습관이 필요한가요?

답:

---
# 5️⃣ Step 5. 레이아웃 — `Row`, `Column`, `Tab`

- `gr.Row()` 안의 컴포넌트는 **가로**로 배치
- `gr.Column()` 안의 컴포넌트는 **세로**로 배치
- `gr.Tab("이름")` 은 **탭**으로 화면 분리

In [ ]:
gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 🪟 좌우 분할 레이아웃")

    with gr.___1___():                  # 빈칸 1 : 가로 배치 컨테이너
        with gr.___2___():              # 빈칸 2 : 세로 배치 컨테이너 (왼쪽)
            gr.Markdown("### 왼쪽 (입력)")
            left_input = gr.Textbox(label="질문")
            left_btn   = gr.Button("전송", variant="primary")

        with gr.Column():
            gr.Markdown("### 오른쪽 (출력)")
            right_output = gr.Textbox(label="답변", lines=5, interactive=False)

    left_btn.click(fn=lambda x: f"답변: {x}", inputs=___3___, outputs=___4___)   # 빈칸 3, 4

demo.launch()

In [ ]:
gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 📑 탭으로 분리된 앱")

    with gr.___1___("💬 채팅"):          # 빈칸 1
        gr.Textbox(label="메시지")

    with gr.Tab("⚙️ 설정"):
        gr.Slider(0, 2, value=0.7, label="Temperature")
        gr.Dropdown(["gpt-4o-mini", "gpt-4o"], value="gpt-4o-mini", label="모델")

    with gr.Tab("📊 사용량"):
        gr.Markdown("토큰 사용량 차트 자리")

demo.launch()

**✍️ 개념 퀴즈** — "왼쪽엔 입력, 오른쪽엔 출력" 구조를 만들 때 Row와 Column은 각각 몇 개 필요한가요?

답:

---
# 6️⃣ Step 6. 이벤트 핸들러 — `.click`, `.submit`, `.change`

| 이벤트 | 트리거 |
|---|---|
| `.click` | ___1___ |
| `.submit` | ___2___ |
| `.change` | ___3___ |

(표 빈칸 1~3을 먼저 적어보세요)

답: 1) ______ 2) ______ 3) ______

In [ ]:
gr.close_all()

with gr.Blocks() as demo:
    input_box  = gr.Textbox(label="질문", placeholder="입력 후 엔터 또는 전송 버튼")
    submit_btn = gr.Button("전송", variant="primary")
    output_box = gr.Textbox(label="답변", interactive=False)

    def process(question):
        return f"질문 '{question}'에 대한 답변입니다."

    submit_btn.___1___(fn=process, inputs=input_box, outputs=output_box)   # 빈칸 1 : 버튼 클릭
    input_box.___2___(fn=process, inputs=input_box, outputs=output_box)    # 빈칸 2 : 엔터

demo.launch()

**✍️ 개념 퀴즈** — 챗봇에서 `.click`과 `.submit`을 **둘 다** 연결하는 이유는?

답:

---
# 7️⃣ Step 7. LangChain 결합 3가지 패턴

### 🔹 패턴 1. Interface + 단순 LLM 호출

In [ ]:
def ask_llm(question: str) -> str:
    return llm.___1___(question).___2___      # 빈칸 1 : 호출 메서드 / 빈칸 2 : AIMessage에서 텍스트 꺼내기

gr.close_all()

demo = gr.Interface(
    fn=ask_llm,
    inputs=gr.Textbox(label="질문"),
    outputs=gr.Textbox(label="답변", lines=8),
    title="🤖 단순 LLM 챗봇",
)
demo.launch()

### 🔹 패턴 2. Blocks + 페르소나 드롭다운
드롭다운 값으로 **system 메시지**가 바뀝니다. (단발 질문 — 메모리 없음)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

PERSONAS = {
    "꼼꼼한 분석가": "당신은 데이터를 꼼꼼히 분석하는 시니어 분석가입니다. 수치와 근거를 중시합니다.",
    "톡톡 튀는 마케터": "당신은 트렌드를 읽는 마케터입니다. 활기차고 창의적인 톤으로 답하세요.",
    "친한 친구": "당신은 사용자의 절친한 친구입니다. 반말로 편하게 답하세요.",
}

persona_prompt = ChatPromptTemplate.from_messages([
    ("system", "{___1___}"),          # 빈칸 1 : 페르소나 문구가 들어갈 변수명
    ("human", "{___2___}"),           # 빈칸 2 : 사용자 질문 변수명
])
persona_chain = persona_prompt ___3___ llm      # 빈칸 3 : LCEL 연결 연산자

def chat_with_persona(persona_name: str, question: str) -> str:
    result = persona_chain.invoke({
        "persona_msg": PERSONAS[___4___],     # 빈칸 4
        "question": question,
    })
    return result.content

> ⚠️ 위 코드는 일부러 **변수명이 어긋나게** 빈칸을 설계했습니다. 빈칸 1에 들어갈 이름과 `invoke({...})`의 키가 **정확히 일치**해야 합니다. (일치하지 않으면 `KeyError`)

In [ ]:
gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 🎭 페르소나 챗봇 v1")

    with gr.Row():
        with gr.Column():
            persona_dd = gr.Dropdown(choices=list(PERSONAS.___1___()), value="친한 친구", label="페르소나")   # 빈칸 1 : dict의 키 목록
            question_box = gr.Textbox(label="질문", lines=3)
            submit_btn = gr.Button("전송", variant="primary")
        with gr.Column():
            answer_box = gr.Textbox(label="답변", lines=10, interactive=False)

    submit_btn.click(chat_with_persona, [___2___, ___3___], answer_box)      # 빈칸 2, 3 (함수 인자 순서!)
    question_box.___4___(chat_with_persona, [persona_dd, question_box], answer_box)   # 빈칸 4

demo.launch()

**✍️ 예측 퀴즈** — 패턴 2 챗봇에 `내 이름은 진환이야` → `내 이름이 뭐였지?` 를 연달아 보내면 기억할까요? 그 이유는? (프롬프트에 무엇이 없나요?)

답:

### 🔹 패턴 3. ChatInterface + 메모리 체인 ⭐

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1) 대화 기록이 들어갈 자리가 있는 프롬프트
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    ___1___("history"),                 # 빈칸 1 : 대화 기록 자리 표시자
    ("human", "{question}"),
])

# 2) 세션 저장소
store = {}
def get_session_history(session_id):
    if session_id ___2___ store:                         # 빈칸 2 : 없으면 (not in / in 중 고르기)
        store[session_id] = ___3___()                    # 빈칸 3 : 메모리에 저장하는 기록 객체
    return store[session_id]

# 3) 체인을 메모리로 감싸기
chain_with_memory = ___4___(                             # 빈칸 4 : 메모리 래퍼 클래스
    memory_prompt | llm,
    get_session_history,
    input_messages_key="___5___",                        # 빈칸 5 : 사용자 입력 변수명
    history_messages_key="___6___",                      # 빈칸 6 : 위 Placeholder 이름과 동일
)

In [ ]:
def chat_fn(message, history):
    result = chain_with_memory.invoke(
        {"question": message},
        config={"___1___": {"session_id": "default"}},   # 빈칸 1 : config의 키
    )
    return result.___2___                                 # 빈칸 2

gr.close_all()
demo = gr.ChatInterface(fn=chat_fn, title="🤖 메모리 챗봇")
demo.launch()

**✍️ 개념 퀴즈**
1. Gradio의 `history`와 LangChain의 `store`는 같은 저장소인가요? 각각의 역할은?
2. `history_messages_key`가 `MessagesPlaceholder`의 이름과 다르면 어떤 일이 생길까요?

답:

---
# 🎁 Step 8. 도전: 페르소나 + 메모리 + Blocks 완전체

`gr.Chatbot` 에 대화를 표시하고, 드롭다운으로 페르소나를 바꾸며, 메모리로 이전 대화를 기억하는 챗봇입니다.

In [ ]:
PERSONAS_FULL = {
    "꼼꼼한 분석가": "당신은 데이터를 꼼꼼히 분석하는 시니어 분석가입니다. 수치와 근거를 중시합니다.",
    "톡톡 튀는 마케터": "당신은 트렌드 마케터입니다. 활기차고 창의적으로 답하세요.",
    "친한 친구": "당신은 사용자의 절친한 친구입니다. 반말로 편하게 답하세요.",
}

full_prompt = ChatPromptTemplate.from_messages([
    ("system", "{___1___}"),                 # 빈칸 1 : 페르소나 변수명
    ___2___("history"),                      # 빈칸 2
    ("human", "{question}"),
])

full_store = {}
def get_full_history(session_id):
    if session_id not in full_store:
        full_store[session_id] = InMemoryChatMessageHistory()
    return full_store[session_id]

full_chain = RunnableWithMessageHistory(
    full_prompt | llm,
    get_full_history,
    input_messages_key="question",
    history_messages_key="history",
)

def respond(message, history, persona_name):
    result = full_chain.invoke(
        {"persona": PERSONAS_FULL[___3___], "question": message},     # 빈칸 3
        config={"configurable": {"session_id": "___4___"}},            # 빈칸 4 : 세션 id ("1" 사용)
    )
    answer = result.content

    # gr.Chatbot(type="messages") 형식
    history = history + [
        {"role": "___5___", "content": message},                      # 빈칸 5
        {"role": "___6___", "content": answer},                       # 빈칸 6
    ]
    return "", history        # 첫 반환값 "" = 입력창 비우기

gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 🎭 페르소나 챗봇 (메모리 포함)")
    persona_dd = gr.Dropdown(choices=list(PERSONAS_FULL.keys()), value="친한 친구", label="페르소나")
    chatbot  = gr.Chatbot(label="대화창", height=400, type="messages")
    msg_box  = gr.Textbox(label="메시지", placeholder="입력 후 엔터")
    send_btn = gr.Button("전송", variant="primary")

    send_btn.click(respond, [msg_box, chatbot, persona_dd], [___7___, ___8___])   # 빈칸 7, 8 : outputs
    msg_box.submit(respond, [msg_box, chatbot, persona_dd], [msg_box, chatbot])

demo.launch()

### 🗑 대화 초기화 — 3가지 전략

| 전략 | 화면(`chatbot`) | 메모리(`full_store`) |
|---|:---:|:---:|
| A | ✅ 비움 | ❌ 그대로 |
| B | ❌ 그대로 | ✅ 비움 |
| C ⭐ | ✅ 비움 | ✅ 비움 |

전략 **C**를 완성하세요.

In [ ]:
def reset_both():
    if "1" ___1___ full_store:          # 빈칸 1 : 세션이 있으면 (in / not in)
        full_store["1"].___2___()       # 빈칸 2 : 저장된 메시지 비우기
    return ___3___                      # 빈칸 3 : 화면(chatbot)을 빈 리스트로

gr.close_all()

with gr.Blocks() as demo:
    persona_dd = gr.Dropdown(choices=list(PERSONAS_FULL.keys()), value="친한 친구", label="페르소나")
    chatbot  = gr.Chatbot(label="대화창", height=400, type="messages")
    msg_box  = gr.Textbox(label="메시지")
    with gr.Row():
        send_btn  = gr.Button("전송", variant="primary")
        reset_btn = gr.Button("🆑 둘 다 비우기", variant="___4___")      # 빈칸 4 : 빨간색 버튼

    send_btn.click(respond, [msg_box, chatbot, persona_dd], [msg_box, chatbot])
    msg_box.submit(respond, [msg_box, chatbot, persona_dd], [msg_box, chatbot])
    reset_btn.click(fn=reset_both, inputs=___5___, outputs=chatbot)      # 빈칸 5 : 입력이 없을 때

demo.launch()

**✍️ 최종 퀴즈** — A(화면만 비움)와 B(메모리만 비움)를 눌렀을 때 각각 어떤 *상태 불일치*가 생기나요? C만 정상인 이유는?

답:

---
# ✅ 정답 (다 풀고 나서 확인하세요!)

> 위로 스크롤하지 않고 풀어본 뒤 열어보는 걸 추천합니다.

### Step 1
1 `close_all` / 2 `fn` / 3 `Textbox` / 4 `launch`
- 퀴즈: `"text"`는 단축 표기(기본 설정). 컴포넌트 객체를 넘기면 `label`, `placeholder`, `lines` 등을 세밀하게 제어 가능.

### Step 2
**메아리 봇**: 1 `message` / 2 `history` / 3 `ChatInterface`

**제곱 봇**: 1 `try` / 2 `int` / 3 `**` / 4 `except`
- 퀴즈: `ValueError`. 사용자는 숫자가 아닌 문자를 입력할 수 있으므로 변환 실패를 잡아 안내 문구를 반환해야 앱이 죽지 않음.

### Step 3
핵심 3가지: 1) `Blocks` 2) 변수 3) `click`

코드: 1 `Blocks` / 2 `interactive` / 3 `variant` / 4 `click` / 5 `input_box` / 6 `output_box`
- 퀴즈: 컴포넌트 **객체(변수)** 자체를 넘기고, Gradio가 실행 시점의 값을 읽어 함수에 전달.

### Step 4
1 `lines` / 2 `Dropdown` / 3 `choices` / 4 `Slider` / 5 `Radio` / 6 `Checkbox` / 7 `stop` / 8 `매개변수`

조용한 버그: 1 `name_box` / 2 `age_box`
- 퀴즈: 에러가 안 나고 타입만 맞으면 그대로 실행되기 때문. "에러 없음 = 정상"이 아니라 **결과가 기대대로인지** 직접 확인하는 습관이 필요.

### Step 5
Row/Column: 1 `Row` / 2 `Column` / 3 `left_input` / 4 `right_output`

Tab: 1 `Tab`
- 퀴즈: Row 1개 + Column 2개.

### Step 6
표: 1 버튼 클릭 / 2 텍스트박스에서 엔터 / 3 값이 바뀔 때

코드: 1 `click` / 2 `submit`
- 퀴즈: 사용자가 전송 버튼을 누르든 엔터를 치든 같은 동작을 하게 하려고.

### Step 7
**패턴 1**: 1 `invoke` / 2 `content`

**패턴 2** (프롬프트): 1 `persona_msg` / 2 `question` / 3 `|` / 4 `persona_name`

**패턴 2** (UI): 1 `keys` / 2 `persona_dd` / 3 `question_box` / 4 `submit`
- 예측 퀴즈: 기억 못함. 프롬프트에 `MessagesPlaceholder`(history 자리)가 없고 메모리 래퍼도 없어서 매 호출이 독립적.

**패턴 3** (체인): 1 `MessagesPlaceholder` / 2 `not in` / 3 `InMemoryChatMessageHistory` / 4 `RunnableWithMessageHistory` / 5 `question` / 6 `history`

**패턴 3** (chat_fn): 1 `configurable` / 2 `content`
- 퀴즈 1: 다른 저장소. `history`는 화면 표시용(Gradio), `store`는 LLM이 참고하는 실제 메모리(LangChain).
- 퀴즈 2: `history_messages_key` 가 Placeholder 이름과 다르면 기록이 프롬프트에 채워지지 않아 오류가 나거나 기억하지 못함.

### Step 8
1 `persona` / 2 `MessagesPlaceholder` / 3 `persona_name` / 4 `1` / 5 `user` / 6 `assistant` / 7 `msg_box` / 8 `chatbot`

초기화 C: 1 `in` / 2 `clear` / 3 `[]` / 4 `stop` / 5 `None`
- 최종 퀴즈: A는 화면만 비어 있고 챗봇은 여전히 기억함, B는 화면엔 대화가 남아 있는데 챗봇은 잊음. **UI와 서버 상태는 별개 저장소**라서 둘 다 비워야 사용자가 보는 것과 챗봇이 아는 것이 일치함.

---
# 📦 (기존 내용 보존) 이전에 적어둔 셀

# 기본 구조 5단계

### 사용할 라이브러리들

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import PydanticOutputParser